# Multimodal MCI Diagnosis & Stability Prediction System
## Thesis Demonstration — End-to-End Pipeline

---

This notebook demonstrates the complete prediction pipeline for a patient record built
from a **real scanned/handwritten clinical report**, combined with representative MRI/PET
demo scans:

| Step | Module | Output |
|------|--------|--------|
| 1 | **Chandra OCR** | Raw text from the clinical report image |
| 1b | **OCR Post-Processing** | Cleaned text (abbreviations, dates, spelling) |
| 2 | **LLM Feature Extraction** (Ministral-3B via Ollama) | Structured clinical feature dictionary |
| 3 | **MRI 3D-CNN Ensemble** (7 folds, early-fusion) | MTA atrophy status |
| 4 | **PET 3D-CNN Ensemble** (7 folds, late-fusion) | Amyloid positivity status |
| 5 | **Feature Integration** | Complete patient record (clinical + imaging) |
| 6 | **LightGBM Diagnosis Classifier** | CN / MCI / Dementia  (93 % F1) |
| 7 | **RNN Stability Predictor** | CN / MCI\_stable / MCI\_converting  (83 % acc) |

**Requires a GPU runtime** (e.g. Kaggle T4) to load Chandra OCR and run Ministral-3B at a
reasonable speed.


In [ ]:
import sys, warnings, pickle
import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

warnings.filterwarnings('ignore')
torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device  : {DEVICE}')
print(f'PyTorch : {torch.__version__}')


---
## Demo Patient

This run is driven by a **real scanned or handwritten clinical report**. Set the image
path in Step 1 below — Chandra OCR reads the report, an LLM (Ministral-3B) extracts
structured clinical features from the raw text, and those features are combined with
MRI + PET imaging biomarkers from a representative demo scan pair to produce a diagnosis
and a stability prediction.


---
## Step 1 — Clinical Report OCR (Chandra VLM)

The clinical report is often scanned as **multiple page images** (one photo per page,
sometimes a sentence continues from one page to the next). Each page is passed through
**Chandra OCR** (`datalab-to/chandra-ocr-2`), a vision-language model, and the per-page
transcriptions are concatenated **in page order** into a single raw text.

**Before running:** set `REPORT_IMAGES_PATH` in the cell below to the folder containing
the report's page images (e.g. a path under `/kaggle/input/...`), or to a single image
file if the report is only one page. Pages are ordered by filename — name them so that
sorting the filenames gives the correct reading order (e.g. `page_01.jpg`, `page_02.jpg`,
... or `1.jpg`, `2.jpg`, ...).


In [ ]:
# ── Install Chandra OCR ───────────────────────────────────────────────────────
import subprocess, sys

print("Installing Chandra OCR (datalab-to/chandra-ocr-2) ...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "chandra-ocr[hf]"])
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyspellchecker"])

import torch
print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
    print(f"VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch        : {torch.__version__}")


In [ ]:
# ── Load Chandra OCR model ───────────────────────────────────────────────────
from transformers import AutoModelForImageTextToText, AutoProcessor
from chandra.model.hf import generate_hf
from chandra.model.schema import BatchInputItem
from chandra.output import parse_markdown
from PIL import Image

print("Loading Chandra OCR on GPU (bfloat16)...")
chandra_model = AutoModelForImageTextToText.from_pretrained(
    "datalab-to/chandra-ocr-2",
    dtype=torch.bfloat16,
    device_map="auto",
)
chandra_model.eval()
chandra_model.processor = AutoProcessor.from_pretrained("datalab-to/chandra-ocr-2")
chandra_model.processor.tokenizer.padding_side = "left"

print("✓ Chandra OCR loaded")
if torch.cuda.is_available():
    print(f"  VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")


In [ ]:
# ── Run Chandra OCR on the clinical report page(s) ───────────────────────────
import re
from pathlib import Path

# ====>  EDIT THIS: folder of scanned report pages, or a single image path  <====
REPORT_IMAGES_PATH = "/kaggle/input/your-dataset/report_pages"

MAX_IMAGE_DIM = 1536   # resize long side to keep inference memory bounded
IMAGE_EXTS    = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp", ".webp"}


def _natural_key(path: Path):
    # so "page_2.jpg" sorts before "page_10.jpg"
    return [int(tok) if tok.isdigit() else tok.lower() for tok in re.split(r'(\d+)', path.stem)]


def _list_report_pages(path: str) -> list:
    p = Path(path)
    if p.is_file():
        return [p]
    pages = sorted((f for f in p.iterdir() if f.suffix.lower() in IMAGE_EXTS), key=_natural_key)
    if not pages:
        raise FileNotFoundError(f"No image files found in {p}")
    return pages


def _resize_for_chandra(img: Image.Image, max_dim: int = MAX_IMAGE_DIM) -> Image.Image:
    w, h = img.size
    longest = max(w, h)
    if longest <= max_dim:
        return img
    scale = max_dim / longest
    return img.resize((max(1, int(w * scale)), max(1, int(h * scale))), Image.LANCZOS)


def _markdown_to_lines(raw_markdown: str) -> str:
    markdown = parse_markdown(raw_markdown)
    lines = []
    for line in markdown.splitlines():
        line = re.sub(r'^#+\s*', '', line.strip())
        line = re.sub(r'^[-*•]\s*', '', line)
        if line:
            lines.append(line)
    return ' '.join(lines)


def run_chandra_ocr(image_path) -> str:
    img = Image.open(image_path).convert("RGB")
    img = _resize_for_chandra(img)
    batch = [BatchInputItem(image=img, prompt_type="ocr_layout")]
    try:
        result = generate_hf(batch, chandra_model)[0]
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        img = _resize_for_chandra(img, max_dim=MAX_IMAGE_DIM // 2)
        batch = [BatchInputItem(image=img, prompt_type="ocr_layout")]
        result = generate_hf(batch, chandra_model)[0]
    return _markdown_to_lines(result.raw)


report_pages = _list_report_pages(REPORT_IMAGES_PATH)
print(f"Found {len(report_pages)} report page(s), in reading order:")
for p in report_pages:
    print(f"   - {p.name}")

ocr_pages = []
for i, page_path in enumerate(report_pages, 1):
    print(f"\nRunning Chandra OCR on page {i}/{len(report_pages)}: {page_path.name} ...")
    page_text = run_chandra_ocr(page_path)
    print(page_text[:300] + (" ..." if len(page_text) > 300 else ""))
    ocr_pages.append(page_text)

ocr_raw = "\n".join(ocr_pages)   # concatenated, following page order

print("\n" + "=" * 62)
print(f"  RAW OCR OUTPUT  —  Chandra VLM  ({len(report_pages)} page(s) concatenated)")
print("=" * 62)
print(ocr_raw)


---
## Step 1b — OCR Post-Processing

Raw OCR output from handwritten French clinical notes contains merged words, malformed
dates, clinical abbreviations and spelling errors. The same post-processing pipeline used
during model development (medical-dictionary lookup + fuzzy matching + French spell
correction) is applied before the text is sent to the LLM.


In [ ]:
# ── OCR post-processing pipeline (medical dictionaries + cleaning) ───────────
import re
from difflib import SequenceMatcher
from typing import List, Dict, Tuple

ABBREV_TXT = """alzheimer,alz,az,alzmr,alzr
ampoule,amp,ampl,emp,empl
ampoules,amps,ampls,emps,empls
année moyenne,am
année primaire,ap
année secondaire,as
antécédent,ancd,antcd,antc,atcd,atcs,atd,ant
antécédents familiaux,antcdf,antcf,antdf,atcdf,atdf,antf
antécédents personnels et familiaux,antfp,antcdfp,atcdfp,antcfp,antfp,antcdfp,atcdfp,antcfp,antpf,antfp
antécédents personnels,antcdp,antcp,antdp,atcdp,atdp,antp
athymil,athl
avec,ac
beaucoup,BCP
biologique,bio
cholestérol,CHOL
cm,cmcomportement,cmpt,cmprt,cmprtmn,cpt,comport,compt,cprt,cpt
comprimé,CP
conduite à tenir,cat
consultation,cons
cortico sous cortical,csc
cérébrale,cereb,cerl,crl
dans,dn,dns
diabète de type 2,dnid
diagnostic,diag
droite,drt,drte
décédé,dcd
examen,ex,exm,exmn
familiaux,f
fns,fns
fois par jour,xj,x/j
fois,x
gauche,gche,gch
glucose,glu
glycémie,gly
goutte,gtt,gtte
gouttes,gtts,gttes
grand,gd
histoire de la maladie,hdm
hypertension artérielle,hta
indiçage,indice,indiçe
insuffisance rénale,ir
jour,jr,j
léger,lg
malade,mld,md,madie
maladie d’Alzheimer,ma
maladie,mde,mdie
mautif de consultation,mc
mnésique,mnes,mnsq
modéré,mod
mémoire,mem,mmr
même,mm
neurologique,neuro
neuropsychologie,neuropsy
niveau intellectuel,ni
niveau,niv
normal,nml
par jour,/j
pendant,pdt
personels,p
pour,pr
présente,pste
psychologue,psycho,psy
quelque,qlq,qqs
rendez-vous,rdv
rien à signaler,ras
sous traitement,s/trt
sous,s,s/
stable,stbl
stade,std
système nerveux central,snc
sévère,sev,svr
tdm cérébrale,tdmc
temps,tp
test,tst
tm,tm
traitement,trt,trtm,trtmn
trouble cognitif,tc
trouble neurocognitif,tnc
troubles,trbl,tbl,tb,trb,tbls,tbs,trbs,trbls
vitamine E,vite
vitamine,vit"""

MEDICATIONS_TXT = """Aricept
aspirine
Aspégic
atacond
Athymil
Aténor
B
Deroxat
Digoxine
donecept
donépézil
Dépakine
E
Ebixa
Exelon
Furosémide
gabatrex
galantamine
Glucophage
Largactil
levocarb
Lévothyrox
mag
micardis
mémantine
Risperdal
rivastigmine
rowasa
synosia
Tensoprel
tolvon
vitamag
vitamine
zoloft"""

TERMS_TXT = """aggravation
agitation
agnosie
agraphie
agressivité
alexie
alzheimer
amimique
amnésie
ampoule
antécédent
antérograde
aphasie
apraxie
artérielle
atrophie
attention
audiovisuel
auditif
bilan
biologique
blanche
cardiopathie
cerveau
clonique
cognitif
comportement
comprimé
compréhension
concentration
confusion
conscience
conscient
consultationcoopérant
cortex
cérébral
diabète
dyslipidémie
dysthyroïdie
déficie
dégradation
démence
démentiel
dépendance
dépressif
dépression
désinvestissement
désorientation
détérioration
encéphalopathie
eeg
ecg
examen
fonction
frontal
gestuel
goutte
hippocampe
hippocampique
hydrocéphalie
hypertension
hypothyroïdie
imagerie
incohérent
indifférence
insomniaque
insomnie
instabilité
insuffisance
intellectuel
IRM
irritabilité
irritabilité
isolement
langage
leucoaraïose
lobe
léger
légèrement
lésion
mg
majeur
malade
maladie
mixte
ml
mmhg
mnésique
modéré
mémoire
neurocognitif
neurodégénérative
neurologique
neuropsychologique
ng
niveau
nécessitant
orientation
orienté
oublis
pariétal
patient
persécution
progressif
prédominance
prédominant
présente
préservé
psychiatrique
psychologique
psychologue
rendez-vous
rigidité
rénal
scanner
sommeil
spatiale
spontanément
spychiatre
stabilisation
stade
stationnaire
substance
syndrome
sévère
TDM
temporal
temporel
temporospatiale
tendance
traitement
troubles
vasculaire
ventricule
épilepsie
épisodique
évaluation
évolution"""

TESTS_TXT = """MMSE
MoCA
l'horloge
IADL
PSMS
ADL
GDS"""


def _load_abbreviations(raw: str) -> Dict[str, str]:
    d = {}
    for line in raw.strip().splitlines():
        line = line.strip()
        if not line or ',' not in line:
            continue
        parts = [p.strip().lower() for p in line.split(',')]
        for variant in parts[1:]:
            d[variant] = parts[0]
    return d


def _load_list(raw: str) -> List[str]:
    return [line.strip().lower() for line in raw.strip().splitlines() if line.strip()]


abbreviations = _load_abbreviations(ABBREV_TXT)
medications   = _load_list(MEDICATIONS_TXT)
terms         = _load_list(TERMS_TXT)
tests         = _load_list(TESTS_TXT)
print(f"Loaded {len(abbreviations)} abbreviations, {len(medications)} medications, "
      f"{len(terms)} terms, {len(tests)} tests")


def split_mixed_words(text: str) -> str:
    text = re.sub(r'(\d+)([a-zàâäéèêëïîôöœçñ]+)', r'\1 \2', text, flags=re.IGNORECASE)
    text = re.sub(r'([a-zàâäéèêëïîôöœçñ]+)(\d+)', r'\1 \2', text, flags=re.IGNORECASE)
    return text


def normalize_dates(text: str) -> str:
    pattern = r'(\d{1,2})\s*[/:\-.\'\s](\d{1,2})\s*[/:\-.\'\s](\d{2,4})'
    def _rep(m):
        day, month, year = m.group(1).zfill(2), m.group(2).zfill(2), m.group(3)
        if len(year) == 2:
            year = '20' + year
        return f"{day}/{month}/{year}\n"
    return re.sub(pattern, _rep, text)


def replace_abbreviations(text: str, abbrev_dict: Dict[str, str]) -> Tuple[str, set]:
    words = text.split()
    result, corrected_indices = [], set()
    for i, word in enumerate(words):
        word_clean = re.sub(r"[^\w'/]", '', word.lower())
        if word_clean in abbrev_dict:
            result.append(abbrev_dict[word_clean])
            corrected_indices.add(i)
        else:
            result.append(word)
    return ' '.join(result), corrected_indices


def find_closest_match(word: str, reference_list: List[str], threshold: float = 0.75) -> Tuple[str, bool]:
    word_lower = word.lower()
    if word_lower in reference_list:
        return word_lower, True
    best_match, best_ratio = None, threshold
    for ref in reference_list:
        ratio = SequenceMatcher(None, word_lower, ref).ratio()
        if ratio > best_ratio:
            best_ratio, best_match = ratio, ref
    return (best_match, True) if best_match else (word, False)


def apply_fuzzy_matching(text: str, medications: List[str], terms: List[str], tests: List[str]) -> Tuple[str, set]:
    words = text.split()
    result, corrected_indices = [], set()
    for i, word in enumerate(words):
        m, ok = find_closest_match(word, medications, 0.75)
        if ok: result.append(m); corrected_indices.add(i); continue
        m, ok = find_closest_match(word, terms, 0.75)
        if ok: result.append(m); corrected_indices.add(i); continue
        m, ok = find_closest_match(word, tests, 0.75)
        if ok: result.append(m); corrected_indices.add(i); continue
        result.append(word)
    return ' '.join(result), corrected_indices


def correct_french_spelling(text: str, skip_indices: set = None,
                             reference_lists: List[List[str]] = None) -> str:
    if skip_indices is None:
        skip_indices = set()
    if reference_lists is None:
        reference_lists = []
    reference_words = {w.lower() for lst in reference_lists for w in lst}
    try:
        from spellchecker import SpellChecker
        spell = SpellChecker(language='fr')
        words = text.split()
        corrected = []
        for i, word in enumerate(words):
            if i in skip_indices:
                corrected.append(word); continue
            word_lower = word.lower()
            word_clean = re.sub(r'[^\w]', '', word_lower)
            if word_clean in reference_words:
                corrected.append(word); continue
            if not re.search(r'[a-zàâäéèêëïîôöœçñ]', word, re.IGNORECASE):
                corrected.append(word); continue
            if word_clean not in spell:
                closest = spell.correction(word_clean)
                corrected.append(closest if closest else word)
            else:
                corrected.append(word)
        return ' '.join(corrected)
    except ImportError:
        return text


def clean_text(text: str) -> str:
    return re.sub(r'\s+', ' ', text).strip()


def remove_noise(text: str, noise_patterns: List[str]) -> str:
    for pat in noise_patterns:
        try:
            text = re.sub(pat, '', text, flags=re.IGNORECASE)
        except re.error:
            text = text.replace(pat, '')
    return re.sub(r'\s+', ' ', text).strip()


NOISE_PATTERNS = ['MouMNi', 'Moumni', 'moumni']


def postprocess_plain_text(raw_text: str) -> str:
    text = split_mixed_words(raw_text)
    text = normalize_dates(text)
    text, abbrev_idx = replace_abbreviations(text, abbreviations)
    text, fuzzy_idx  = apply_fuzzy_matching(text, medications, terms, tests)
    text = correct_french_spelling(
        text,
        skip_indices=abbrev_idx | fuzzy_idx,
        reference_lists=[medications, tests, ['CM', 'TM', 'ro']],
    )
    text = remove_noise(text, NOISE_PATTERNS)
    text = clean_text(text)
    return text


ocr_cleaned = postprocess_plain_text(ocr_raw)

print("=" * 62)
print("  RAW OCR  vs  CLEANED (post-processed)")
print("=" * 62)
print("RAW    :", ocr_raw[:600])
print()
print("CLEANED:", ocr_cleaned[:600])


---
## Step 2 — LLM Feature Extraction (Ministral-3B via Ollama)

The cleaned OCR text is sent to a locally-served **Ministral-3B** model (through Ollama)
with a structured extraction prompt. The model returns a JSON object containing the
ADNI-compatible clinical features used by the downstream classifiers. Fields that are not
present in the report come back as `null` and are imputed later from training-set medians,
exactly as the production system does for missing data.


In [ ]:
# ── Install & start Ollama, pull Ministral-3B ─────────────────────────────────
import subprocess, sys, time

print("Installing Ollama ...")
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, capture_output=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ollama"])

print("Starting Ollama server ...")
ollama_process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(5)

import ollama

print("Pulling ministral-3:3b (first run can take a few minutes) ...")
try:
    ollama.pull("ministral-3:3b")
except Exception as e:
    print(f"  ollama.pull() failed ({e}), retrying via CLI ...")
    subprocess.run(["ollama", "pull", "ministral-3:3b"], capture_output=True)

test = ollama.chat(model="ministral-3:3b",
                    messages=[{"role": "user", "content": "Reply with OK only."}],
                    stream=False)
print(f"✓ Ministral-3B ready  —  test reply: {test['message']['content'].strip()}")


In [ ]:
# ── Structured feature-extraction prompt ──────────────────────────────
import json, re

EXTRACTION_PROMPT = """Tu es un assistant clinique specialise en neuropsychologie algerienne.
Analyse le compte-rendu medical francais ci-dessous et renvoie UNIQUEMENT un objet JSON
strict avec exactement les champs suivants. Utilise `null` si une valeur est absente du
texte --- n'invente jamais de valeur.

{
  "age": null,
  "PTGENDER": null,
  "PTHAND": null,
  "PTMARRY": null,
  "PTEDUCAT": null,
  "PTWORK": null,
  "PTNOTRT": null,
  "VISDATE": null,
  "MMSCORE": null,
  "MOCA": null,
  "FAQ": null,
  "MH4CARD": null,
  "MH9ENDO": null,
  "MHPSYCH": null,
  "MH2NEURL": null,
  "MH16SMOK": null,
  "MH14ALCH": null,
  "MOTHDEM": null,
  "FATHDEM": null,
  "SIBDEMENT": null,
  "CMMED": null,
  "CMDOSE": null,
  "KEYMED": null
}

Regles :
- "PTGENDER"  : "Male" ou "Female"
- "PTHAND"    : 1 si droitier, 2 si gaucher, null sinon
- "PTMARRY"   : 1 si marie(e), 2 si veuf/veuve, 3 si divorce(e), 4 si celibataire, null sinon
- "PTEDUCAT"  : annees de scolarite (0 si analphabete)
- "PTWORK"    : description de l'occupation (ex: "retraite", "enseignant"), null sinon
- "PTNOTRT"   : 0 si sous traitement medicamenteux, 1 sinon
- "VISDATE"   : date de visite au format jj/mm/aaaa
- "MMSCORE", "MOCA", "FAQ" : scores totaux uniquement (pas les sous-scores)
- "MH4CARD"   : 1 si HTA / cardiopathie / antecedents cardiovasculaires, null sinon
- "MH9ENDO"   : 1 si diabete / pathologie thyroidienne, null sinon
- "MHPSYCH"   : 1 si antecedents psychiatriques / depression / anxiete, null sinon
- "MH2NEURL"  : 1 si antecedents neurologiques (AVC, epilepsie, Parkinson), null sinon
- "MH16SMOK"  : 1 si tabagisme mentionne, null sinon
- "MH14ALCH"  : 1 si consommation d'alcool mentionnee, null sinon
- "MOTHDEM"   : 1 si demence / Alzheimer chez la mere, null sinon
- "FATHDEM"   : 1 si demence / Alzheimer chez le pere, null sinon
- "SIBDEMENT" : 1 si demence chez un frere ou une soeur, null sinon
- "CMMED"     : premier medicament principal mentionne, null sinon
- "CMDOSE"    : dose du premier medicament (ex: "5mg"), null sinon
- "KEYMED"    : liste de tous les medicaments separes par virgule, null sinon
- Reponds uniquement avec le JSON, sans texte additionnel, sans balises markdown.

Compte-rendu :
```
{text}
```
"""


def parse_llm_json(raw_response: str) -> dict:
    text = re.sub(r'```json\s*', '', raw_response)
    text = re.sub(r'```\s*$', '', text).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r'\{.*\}', text, re.DOTALL)
        if match:
            return json.loads(match.group(0))
        raise ValueError(f"Could not parse LLM JSON output:\n{raw_response}")


def extract_features_with_llm(cleaned_text: str, model: str = "ministral-3:3b") -> dict:
    prompt = EXTRACTION_PROMPT.replace("{text}", cleaned_text)
    response = ollama.chat(model=model, messages=[{"role": "user", "content": prompt}], stream=False)
    raw = response["message"]["content"]
    return parse_llm_json(raw), raw


llm_features, llm_raw_response = extract_features_with_llm(ocr_cleaned)

print("RAW LLM RESPONSE")
print("-" * 45)
print(llm_raw_response)
print()
print("PARSED JSON")
print("-" * 45)
print(json.dumps(llm_features, indent=2, ensure_ascii=False))

In [ ]:
# ── Build the clinical feature dictionary from the LLM output ────────────
features_from_ocr = dict(llm_features)

print("STRUCTURED FEATURES EXTRACTED BY LLM")
print("-" * 45)
groups = {
    "Demographics": ["age", "PTGENDER", "PTHAND", "PTMARRY", "PTEDUCAT", "PTWORK", "PTNOTRT", "VISDATE"],
    "Cognitive":    ["MMSCORE", "MOCA", "FAQ"],
    "Medical Hx":   ["MH4CARD", "MH9ENDO", "MHPSYCH", "MH2NEURL", "MH16SMOK", "MH14ALCH"],
    "Family Hx":    ["MOTHDEM", "FATHDEM", "SIBDEMENT"],
    "Medications":  ["CMMED", "CMDOSE", "KEYMED"],
}
for grp, keys in groups.items():
    print(f"\n  [{grp}]")
    for k in keys:
        v = features_from_ocr.get(k)
        flag = "  (not found in report -> will be imputed)" if v is None else ""
        print(f"    {k:14s}: {v}{flag}")

n_found = sum(1 for v in features_from_ocr.values() if v is not None)
print(f"\n  Features found in report: {n_found} / {len(features_from_ocr)}")

---
## Step 3 — MRI Brain Atrophy Classification

The T1-weighted MRI is skull-stripped, registered to MNI152 space and resampled
to 96×96×96 voxels.  A 7-fold ensemble of **Small3DCNN** models (early-fusion with
demographic MLP) classifies the volume as **Atrophic** or **Healthy**.


In [ ]:
# ── Load & visualise MRI ─────────────────────────────────────────────────────
MRI_SUBJECT = "348321"   # atrophic example
MRI_IMAGE_DIR = Path("MRI - atrophy classification/images")

mri_vol = np.load(MRI_IMAGE_DIR / f"{MRI_SUBJECT}.npy").astype(np.float32)
print(f"Volume shape : {mri_vol.shape}   (96×96×96 voxels, MNI-registered)")
print(f"Intensity    : [{mri_vol.min():.3f}, {mri_vol.max():.3f}]")

mid = [s // 2 for s in mri_vol.shape]
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), facecolor='#0d0d0d')
planes = [
    (mri_vol[mid[0], :, :], "Axial  (z = 48)"),
    (mri_vol[:, mid[1], :], "Coronal (y = 48)"),
    (mri_vol[:, :, mid[2]], "Sagittal (x = 48)"),
]
for ax, (slc, title) in zip(axes, planes):
    im = ax.imshow(np.rot90(slc), cmap='gray', interpolation='bilinear')
    ax.set_title(title, color='white', fontsize=12, pad=8)
    ax.axis('off')
fig.suptitle(f"Subject {MRI_SUBJECT} — T1w MRI (preprocessed: N4 + skull-strip + MNI152)",
             color='white', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("mri_slices.png", dpi=110, bbox_inches='tight', facecolor='#0d0d0d')
plt.show()


In [ ]:
# ── MRI model architecture (must match training) ─────────────────────────────
class Small3DCNNDemo(nn.Module):
    def __init__(self, n_demo_features=3, demo_hidden=16,
                 n_targets=1, dropout=0.3, channels=(16, 32, 64, 128)):
        super().__init__()
        def block(in_c, out_c):
            return nn.Sequential(
                nn.Conv3d(in_c, out_c, 3, padding=1, bias=False),
                nn.BatchNorm3d(out_c), nn.ReLU(inplace=True), nn.MaxPool3d(2))
        layers = [block(1, channels[0])]
        for i in range(len(channels) - 1):
            layers.append(block(channels[i], channels[i + 1]))
        self.backbone     = nn.Sequential(*layers)
        self.global_pool  = nn.AdaptiveAvgPool3d(1)
        self.demo_encoder = nn.Sequential(
            nn.Linear(n_demo_features, demo_hidden), nn.ReLU(inplace=True),
            nn.Linear(demo_hidden, demo_hidden),     nn.ReLU(inplace=True))
        self.dropout = nn.Dropout(dropout)
        self.head    = nn.Linear(channels[-1] + demo_hidden, n_targets)

    def forward(self, x, demo):
        feat = self.global_pool(self.backbone(x)).flatten(1)
        d    = self.demo_encoder(demo)
        return self.head(self.dropout(torch.cat([feat, d], dim=1))).squeeze(1)

CHANNEL_CONFIGS = {
    "narrow_3":   (8,  16,  32),
    "narrow_4":   (8,  16,  32,  64),
    "standard_4": (16, 32,  64, 128),
    "wide_4":     (32, 64, 128, 256),
}
print("Small3DCNNDemo architecture loaded.")


In [ ]:
# ── MRI 7-fold ensemble inference ────────────────────────────────────────────
MRI_FOLD_DIR = Path("MRI - atrophy classification/models")
MRI_CSV_PATH = Path("MRI - atrophy classification/data/data.csv")

df_mri = pd.read_csv(MRI_CSV_PATH)
df_mri["image_id"] = df_mri["image_id"].astype(str).str.replace(".npy", "", regex=False)
row_mri = df_mri[df_mri["image_id"] == MRI_SUBJECT].iloc[0]
AGE_MRI = float(row_mri["AGE"])
SEX_MRI = str(row_mri["PTGENDER"])
EDU_MRI = float(row_mri["PTEDUCAT"])
print(f"Subject {MRI_SUBJECT}  |  AGE={AGE_MRI}  SEX={SEX_MRI}  EDU={EDU_MRI}")

x_mri = torch.from_numpy(mri_vol[None, None]).to(DEVICE)  # (1,1,96,96,96)

mri_probs, mri_thrs = [], []
for k in range(7):
    with open(MRI_FOLD_DIR / f"fold_{k}.pkl", "rb") as f:
        res = pickle.load(f)
    sc  = res["demo_scaler"]; cfg = res["config"]; thr = res["test_results"]["threshold"]
    gender = 1.0 if SEX_MRI.strip().upper().startswith("F") else 0.0
    age_z  = (AGE_MRI - sc["age_mean"]) / sc["age_std"]
    edu_z  = (EDU_MRI - sc["edu_mean"]) / sc["edu_std"]
    demo   = torch.tensor([[gender, age_z, edu_z]], dtype=torch.float32).to(DEVICE)
    m = Small3DCNNDemo(channels=CHANNEL_CONFIGS[cfg["channels_key"]],
                       demo_hidden=cfg["demo_hidden"], dropout=cfg["dropout"]).to(DEVICE)
    m.load_state_dict(torch.load(MRI_FOLD_DIR / f"fold_{k}_best_model.pt",
                                  map_location=DEVICE, weights_only=True))
    m.eval()
    with torch.no_grad():
        p = torch.sigmoid(m(x_mri, demo)).item()
    mri_probs.append(p); mri_thrs.append(thr)
    verdict = "ATROPHIC" if p >= thr else "healthy"
    print(f"  Fold {k}: prob={p:.3f}  thr={thr:.3f}  → {verdict}")
    del m

mri_avg  = float(np.mean(mri_probs))
mri_thr  = float(np.mean(mri_thrs))
MRI_ATROPHY = int(mri_avg >= mri_thr)

print(f"\n  Ensemble avg prob : {mri_avg:.3f}")
print(f"  Mean threshold    : {mri_thr:.3f}")
print(f"  MRI RESULT        : {'ATROPHIC  (MTA=1)' if MRI_ATROPHY else 'HEALTHY  (MTA=0)'}")


---
## Step 4 — PET Amyloid Positivity Classification

The FDG/amyloid PET scan is smoothed (4 mm FWHM), registered to FBP template via SyN,
SUVR-normalised to cerebellar reference (= 1.0), and classified by a 7-fold
**Simple3DCNN\_LateFusion** ensemble.


In [ ]:
# ── Load & visualise PET ─────────────────────────────────────────────────────
try:
    import nibabel as nib
except ImportError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "nibabel", "-q"])
    import nibabel as nib

PET_SUBJECT  = "I225450"   # amyloid-positive example
PET_IMAGE_DIR = Path("PET - amyloid positivity classification/images")

pet_img   = nib.load(str(PET_IMAGE_DIR / f"{PET_SUBJECT}.nii.gz"))
pet_vol   = pet_img.get_fdata(dtype=np.float32)
print(f"PET volume shape : {pet_vol.shape}")
print(f"SUVR range       : [{pet_vol.min():.3f}, {pet_vol.max():.3f}]")

mid_p = [s // 2 for s in pet_vol.shape]
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), facecolor='#0d0d0d')
planes_p = [
    (pet_vol[mid_p[0], :, :], "Axial"),
    (pet_vol[:, mid_p[1], :], "Coronal"),
    (pet_vol[:, :, mid_p[2]], "Sagittal"),
]
for ax, (slc, title) in zip(axes, planes_p):
    im = ax.imshow(np.rot90(slc), cmap='hot', interpolation='bilinear',
                   vmin=0.5, vmax=2.5)
    ax.set_title(title, color='white', fontsize=12, pad=8)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04).ax.yaxis.set_tick_params(color='white')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04).ax.tick_params(colors='white', labelsize=8)
fig.suptitle(f"Subject {PET_SUBJECT} — Amyloid PET  (SUVR, cerebellar ref = 1.0)",
             color='white', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("pet_slices.png", dpi=110, bbox_inches='tight', facecolor='#0d0d0d')
plt.show()


In [ ]:
# ── PET model architecture (must match training) ─────────────────────────────
class Simple3DCNN_LateFusion(nn.Module):
    def __init__(self, channels=(32,64,128,256), fc_units=128,
                 dropout=0.3, kernel_size=3, demo_hidden=16):
        super().__init__()
        blocks, in_ch = [], 1
        for out_ch in channels:
            blocks += [nn.Conv3d(in_ch, out_ch, kernel_size, padding=kernel_size//2, bias=False),
                       nn.BatchNorm3d(out_ch), nn.ReLU(inplace=True), nn.MaxPool3d(2)]
            in_ch = out_ch
        self.encoder     = nn.Sequential(*blocks)
        self.gap         = nn.AdaptiveAvgPool3d(1)
        self.demo_branch = nn.Sequential(nn.Linear(2, demo_hidden), nn.ReLU())
        self.dropout     = nn.Dropout(dropout)
        self.fc          = nn.Linear(in_ch + demo_hidden, fc_units)
        self.head        = nn.Linear(fc_units, 2)

    def forward(self, img, demo):
        x = self.gap(self.encoder(img)).flatten(1)
        x = self.dropout(torch.cat([x, self.demo_branch(demo)], dim=1))
        return self.head(F.relu(self.fc(x)))

PET_CFG = {"channels": (32, 64, 128, 256), "kernel_size": 3,
           "fc_units": 128, "dropout": 0.3, "demo_hidden": 16}
print("Simple3DCNN_LateFusion architecture loaded.")


In [ ]:
# ── PET 7-fold ensemble inference ────────────────────────────────────────────
PET_FOLD_DIR = Path("PET - amyloid positivity classification/models")
PET_CSV_PATH = Path("PET - amyloid positivity classification/data/amy_dataset_final.csv")

df_pet = pd.read_csv(PET_CSV_PATH)
df_pet["image_id_2_str"] = df_pet["image_id_2_str"].astype(str).str.replace(".nii.gz","",regex=False)
row_pet = df_pet[df_pet["image_id_2_str"] == PET_SUBJECT].iloc[0]
AGE_PET = float(row_pet["AGE_AT_SCAN"])
SEX_PET = str(row_pet["SEX"])
print(f"Subject {PET_SUBJECT}  |  AGE={AGE_PET}  SEX={SEX_PET}")

age_z_p = (AGE_PET - df_pet["AGE_AT_SCAN"].mean()) / df_pet["AGE_AT_SCAN"].std()
sex_f_p = 1.0 if SEX_PET.strip().upper() in ("M","MALE","1") else 0.0
demo_pet = torch.tensor([[age_z_p, sex_f_p]], dtype=torch.float32).to(DEVICE)
x_pet = torch.from_numpy(pet_vol).unsqueeze(0).unsqueeze(0).to(DEVICE)

pet_probs, pet_thrs = [], []
for k in range(7):
    ckpt = torch.load(PET_FOLD_DIR / f"fold_{k}.pt", map_location=DEVICE, weights_only=False)
    thr  = ckpt["best_threshold"]
    m = Simple3DCNN_LateFusion(**PET_CFG).to(DEVICE)
    m.load_state_dict(ckpt["state_dict"])
    m.eval()
    with torch.no_grad():
        p = torch.softmax(m(x_pet, demo_pet), dim=1)[0, 1].item()
    pet_probs.append(p); pet_thrs.append(thr)
    verdict = "POSITIVE" if p >= thr else "negative"
    print(f"  Fold {k}: prob={p:.3f}  thr={thr:.3f}  → {verdict}")
    del m

pet_avg      = float(np.mean(pet_probs))
pet_thr      = float(np.mean(pet_thrs))
AMYLOID_STATUS = int(pet_avg >= pet_thr)

print(f"\n  Ensemble avg prob : {pet_avg:.3f}")
print(f"  Mean threshold    : {pet_thr:.3f}")
print(f"  PET RESULT        : {'AMYLOID POSITIVE  (Amy=1)' if AMYLOID_STATUS else 'AMYLOID NEGATIVE  (Amy=0)'}")


---
## Step 5 — Feature Integration

Imaging biomarkers (MTA atrophy + amyloid status) are appended to the
OCR-extracted clinical features, completing the patient record before classification.


In [ ]:
# ── Merge imaging results into clinical feature record ────────────────
patient_record = dict(features_from_ocr)   # copy OCR/LLM-extracted features
patient_record["MTA_ATROPHY"]    = MRI_ATROPHY      # from Step 3
patient_record["AMYLOID_STATUS"] = AMYLOID_STATUS    # from Step 4


def _fmt(v):
    return "N/A (not extracted)" if v is None else str(v)


print("COMPLETE PATIENT RECORD")
print("=" * 50)
summary = {
    "Age":              patient_record.get("age"),
    "Sex":              patient_record.get("PTGENDER"),
    "Education (yr)":   patient_record.get("PTEDUCAT"),
    "Marital status":   patient_record.get("PTMARRY"),
    "Occupation":       patient_record.get("PTWORK"),
    "MMSE":             patient_record.get("MMSCORE"),
    "MoCA":             patient_record.get("MOCA"),
    "FAQ":              patient_record.get("FAQ"),
    "HTA / Cardio":     patient_record.get("MH4CARD"),
    "Diabetes / Endo":  patient_record.get("MH9ENDO"),
    "Psychiatric Hx":   patient_record.get("MHPSYCH"),
    "Neurological Hx":  patient_record.get("MH2NEURL"),
    "Smoking":          patient_record.get("MH16SMOK"),
    "Family dem. (Mo)": patient_record.get("MOTHDEM"),
    "Family dem. (Fa)": patient_record.get("FATHDEM"),
    "Medication":       patient_record.get("CMMED"),
    "MRI Atrophy":      "YES (MTA=1)" if MRI_ATROPHY else "NO  (MTA=0)",
    "Amyloid PET":      "POSITIVE (1)" if AMYLOID_STATUS else "NEGATIVE (0)",
}
for k, v in summary.items():
    abnormal = (
        (k == "MMSE"        and v is not None and v < 24) or
        (k == "MoCA"        and v is not None and v < 26) or
        (k == "FAQ"         and v is not None and v > 5)  or
        (k == "MRI Atrophy" and "YES" in str(v))          or
        (k == "Amyloid PET" and "POSITIVE" in str(v))
    )
    flag = " <- ABNORMAL" if abnormal else ""
    print(f"  {k:22s}: {_fmt(v)}{flag}")
print(f"\n  Total features in record: {len(patient_record)}")

---
## Step 6 — Diagnosis Classification (LightGBM)

The complete patient record is preprocessed (imputation, scaling, missing flags)
and passed to the **LightGBM** classifier (600 trees, 63 leaves, balanced class weights).
**Best model on test set: 93.04% accuracy, 0.9302 macro-F1.**

The system also maintains a soft-vote **neural ensemble** (GRNNet + DeepResNet-SE +
DualStreamDomainNet, ~90% F1) as a complementary predictor.


In [ ]:
# ── Load saved artifacts (Algerian stream) ────────────────────────────────────
DIAG_ROOT = Path("diagnosis classification/diagnosis_models/alg")

diag_arts = {
    "scaler":          joblib.load(DIAG_ROOT / "scaler.pkl"),
    "label_encoder":   joblib.load(DIAG_ROOT / "label_encoder.pkl"),
    "feature_cols":    joblib.load(DIAG_ROOT / "feature_cols.pkl"),
    "global_medians":  joblib.load(DIAG_ROOT / "global_medians.pkl"),
    "string_encoders": joblib.load(DIAG_ROOT / "string_encoders.pkl"),
    "lightgbm":        joblib.load(DIAG_ROOT / "lightgbm.pkl"),
}

# Original label encoding: 0=CN  1=Dementia  2=MCI
DIAG_LABELS = {0: "CN", 1: "Dementia", 2: "MCI"}

print(f"Diagnosis artifacts loaded (algerian stream)")
print(f"  Feature columns : {len(diag_arts['feature_cols'])}")
print(f"  String encoders : {list(diag_arts['string_encoders'].keys())[:5]} …")


In [ ]:
# ── Preprocessing + inference helpers ────────────────────────────────────────
def preprocess_instance(instance_dict, arts):
    feat_cols = arts["feature_cols"]
    glb_med   = arts["global_medians"]
    str_enc   = arts["string_encoders"]
    scaler    = arts["scaler"]
    row = {}
    for col in feat_cols:
        if col.endswith("_missing"):
            base = col[:-len("_missing")]
            row[col] = 1.0 if (base not in instance_dict or
                                pd.isna(instance_dict.get(base, np.nan))) else 0.0
        elif col in str_enc:
            val = str(instance_dict.get(col, "not_recorded"))
            le  = str_enc[col]
            val = val if val in le.classes_ else le.classes_[0]
            row[col] = float(le.transform([val])[0])
        else:
            val = instance_dict.get(col, np.nan)
            row[col] = float(glb_med.get(col, 0.0)) if pd.isna(val) else float(val)
    X_raw = np.array([[row[c] for c in feat_cols]], dtype=np.float32)
    return scaler.transform(X_raw).astype(np.float32)


def predict_diagnosis(instance_dict, arts):
    X     = preprocess_instance(instance_dict, arts)
    probs = arts["lightgbm"].predict_proba(X)[0]
    pred  = int(np.argmax(probs))
    return {"prediction": DIAG_LABELS[pred],
            "class_index": pred,
            "probabilities": {DIAG_LABELS[i]: float(p) for i, p in enumerate(probs)}}


# ── Run prediction ────────────────────────────────────────────────────────────
diag_result = predict_diagnosis(patient_record, diag_arts)

print("\nDIAGNOSIS CLASSIFICATION RESULT")
print("=" * 50)
print(f"  Predicted diagnosis : {diag_result['prediction']}")
print()
print("  Class probabilities :")
for cls, prob in sorted(diag_result["probabilities"].items(), key=lambda x: -x[1]):
    bar = "█" * int(prob * 36)
    print(f"    {cls:10s}  {bar:<36s}  {prob:.3f}")


---
## Step 7 — MCI Stability Prediction (RNN)

The patient's longitudinal visit history (multiple clinic visits) is fed into a
**RecurrentClassifier** (stacked residual RNN + temporal attention, 36-D by-domain PCA input).
The model predicts the MCI trajectory: **CN / MCI\_stable / MCI\_converting**.
**Test accuracy: 83.5%,  macro-F1: 0.79** (algerian stream).


In [ ]:
# ── Stability model architecture (must match training exactly) ───────────────
class TemporalAttention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attn = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, hs, lengths):
        B, T, H = hs.shape
        scores = self.attn(hs).squeeze(-1)
        mask   = torch.arange(T, device=hs.device).unsqueeze(0) < lengths.unsqueeze(1)
        scores = scores.masked_fill(~mask, float("-inf"))
        weights = F.softmax(scores, dim=1).unsqueeze(-1)
        return (weights * hs).sum(dim=1), weights.squeeze(-1)


class ResidualRNNBlock(nn.Module):
    def __init__(self, cell_type, input_size, hidden_size, dropout=0.0):
        super().__init__()
        RNNClass = {"rnn": nn.RNN, "gru": nn.GRU, "lstm": nn.LSTM}[cell_type.lower()]
        self.rnn  = RNNClass(input_size, hidden_size, num_layers=1, batch_first=True)
        self.drop = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(hidden_size)
        self.proj = nn.Linear(input_size, hidden_size) if input_size != hidden_size else nn.Identity()

    def forward(self, x, hx=None):
        out, hx_new = self.rnn(x, hx)
        return self.norm(self.drop(out) + self.proj(x)), hx_new


class RecurrentClassifier(nn.Module):
    def __init__(self, input_size, num_classes, cell_type="rnn",
                 hidden_size=128, num_layers=3, dropout=0.3):
        super().__init__()
        self.input_proj = nn.Linear(input_size, hidden_size)
        self.blocks     = nn.ModuleList([
            ResidualRNNBlock(cell_type, hidden_size, hidden_size, dropout)
            for _ in range(num_layers)])
        self.attention  = TemporalAttention(hidden_size)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(hidden_size // 2, num_classes))

    def forward(self, x, lengths):
        h = self.input_proj(x)
        for blk in self.blocks:
            h, _ = blk(h)
        ctx, _ = self.attention(h, lengths)
        return self.classifier(ctx)

print("RecurrentClassifier architecture loaded.")


In [ ]:
# ── Load stability artifacts + rebuild model ──────────────────────────────────
STAB_ROOT = Path("stability classification/stability_models/alg")

stab_arts = {
    "pca_reducers":  joblib.load(STAB_ROOT / "pca_reducers.pkl"),
    "norm_stats":    joblib.load(STAB_ROOT / "norm_stats.pkl"),
    "feature_cols":  joblib.load(STAB_ROOT / "feature_cols.pkl"),
    "label_encoder": joblib.load(STAB_ROOT / "label_encoder.pkl"),
    "cat_encoders":  joblib.load(STAB_ROOT / "cat_encoders.pkl"),
    "global_means":  joblib.load(STAB_ROOT / "global_means.pkl"),
    "model_config":  joblib.load(STAB_ROOT / "model_config.pkl"),
}

cfg_stab = stab_arts["model_config"]
stab_model = RecurrentClassifier(
    input_size   = cfg_stab["input_size"],
    num_classes  = cfg_stab["num_classes"],
    cell_type    = cfg_stab["cell_type"],
    hidden_size  = cfg_stab["hidden_size"],
    num_layers   = cfg_stab["num_layers"],
    dropout      = cfg_stab["dropout"],
).to(DEVICE)
stab_model.load_state_dict(
    torch.load(STAB_ROOT / "rnn_cfg2_domain_pca.pt", map_location=DEVICE, weights_only=True))
stab_model.eval()

print(f"Stability model loaded  ({sum(p.numel() for p in stab_model.parameters()):,} parameters)")
print(f"  input_size  : {cfg_stab['input_size']} (36-D by-domain PCA)")
print(f"  hidden_size : {cfg_stab['hidden_size']}")
print(f"  num_layers  : {cfg_stab['num_layers']}")
print(f"  classes     : {list(stab_arts['label_encoder'].classes_)}")


In [ ]:
# ── Preprocessing helper ──────────────────────────────────────────────────────
def preprocess_visits_for_inference(visit_list, arts):
    """Convert list of visit dicts to PCA-reduced (1, T, D) tensor."""
    feat_cols  = arts["feature_cols"]
    cat_enc    = arts["cat_encoders"]
    gm         = arts["global_means"]
    mu, std    = arts["norm_stats"]["mu"], arts["norm_stats"]["std"]
    pca_red    = arts["pca_reducers"]

    rows = []
    for visit in visit_list:
        row = []
        for col in feat_cols:
            if col in cat_enc:
                val = str(visit.get(col, "UNKNOWN"))
                le  = cat_enc[col]
                val = val if val in le.classes_ else le.classes_[0]
                row.append(float(le.transform([val])[0]))
            elif col.endswith("_missing"):
                base = col[:-len("_missing")]
                row.append(1.0 if base not in visit or pd.isna(visit.get(base, np.nan)) else 0.0)
            elif col.endswith("_delta"):
                row.append(0.0)
            else:
                val = visit.get(col, np.nan)
                row.append(float(gm.get(col, 0.0)) if pd.isna(val) else float(val))
        rows.append(row)

    X_raw  = np.array(rows, dtype=np.float32)
    X_norm = np.nan_to_num((X_raw - mu) / std, nan=0.0)

    parts = []
    for domain, red in pca_red.items():
        idxs  = [i for i in red["indices"] if i < X_norm.shape[1]]
        if idxs:
            parts.append(red["pca"].transform(X_norm[:, idxs]))
    X_pca = np.hstack(parts).astype(np.float32)
    return torch.tensor(X_pca).unsqueeze(0), torch.tensor([len(visit_list)], dtype=torch.long)


def predict_stability(visit_list, arts, model):
    model.eval()
    X, lengths = preprocess_visits_for_inference(visit_list, arts)
    X = X.to(DEVICE); lengths = lengths.to(DEVICE)
    with torch.no_grad():
        probs = F.softmax(model(X, lengths), dim=1).cpu().numpy()[0]
    pred  = int(np.argmax(probs))
    le    = arts["label_encoder"]
    return {"prediction":    le.classes_[pred],
            "probabilities": {le.classes_[i]: float(p) for i, p in enumerate(probs)},
            "n_visits":      len(visit_list)}

print("Stability inference helpers ready.")


In [ ]:
# ── Longitudinal visit history ────────────────────────────────────────────
# Only the CURRENT visit comes from the real OCR -> LLM -> imaging pipeline above.
# The 3 earlier visits are synthesized as a plausible declining trend so the RNN
# stability model has a full multi-visit trajectory to reason over.
current = dict(patient_record)


def _num(v, default):
    if v is None:
        return default
    try:
        return float(v)
    except (TypeError, ValueError):
        return default


cur_age  = _num(current.get("age"),     74)
cur_mmse = _num(current.get("MMSCORE"), 24)
cur_moca = _num(current.get("MOCA"),    22)
cur_faq  = _num(current.get("FAQ"),      5)

longitudinal_visits = [
    {**current, "age": cur_age - 4,
     "MMSCORE": int(min(cur_mmse + 5, 30)), "MOCA": int(min(cur_moca + 5, 30)),
     "FAQ": int(max(cur_faq - 7, 0)),
     "AMYLOID_STATUS": 0, "MTA_ATROPHY": 0,
     "VISCODE2_num": 0,  "VISDATE_days": 0},        # baseline (4 yrs ago)

    {**current, "age": cur_age - 3,
     "MMSCORE": int(min(cur_mmse + 3, 30)), "MOCA": int(min(cur_moca + 3, 30)),
     "FAQ": int(max(cur_faq - 5, 0)),
     "AMYLOID_STATUS": AMYLOID_STATUS, "MTA_ATROPHY": 0,
     "VISCODE2_num": 12, "VISDATE_days": 365},       # year 1

    {**current, "age": cur_age - 2,
     "MMSCORE": int(min(cur_mmse + 2, 30)), "MOCA": int(min(cur_moca + 2, 30)),
     "FAQ": int(max(cur_faq - 3, 0)),
     "AMYLOID_STATUS": AMYLOID_STATUS, "MTA_ATROPHY": MRI_ATROPHY,
     "VISCODE2_num": 24, "VISDATE_days": 730},       # year 2

    {**current, "VISCODE2_num": 48, "VISDATE_days": 1460},   # current visit
]

print("Longitudinal trajectory (MMSE / MoCA / FAQ):")
print(f"  {'Visit':>6}  {'Yr':>4}  {'MMSE':>5}  {'MoCA':>5}  {'FAQ':>4}")
for i, v in enumerate(longitudinal_visits):
    yr = v["VISDATE_days"] // 365
    mmse_disp = v.get("MMSCORE") if v.get("MMSCORE") is not None else cur_mmse
    moca_disp = v.get("MOCA")    if v.get("MOCA")    is not None else cur_moca
    faq_disp  = v.get("FAQ")     if v.get("FAQ")      is not None else cur_faq
    print(f"  {'BL' if yr==0 else f'Y{yr}':>6}  {yr:>4}  "
          f"{mmse_disp:>5}  {moca_disp:>5}  {faq_disp:>4}")

In [ ]:
# ── Run stability prediction ──────────────────────────────────────────────────
stab_result = predict_stability(longitudinal_visits, stab_arts, stab_model)

print("MCI STABILITY PREDICTION RESULT")
print("=" * 50)
print(f"  Visits analysed     : {stab_result['n_visits']}")
print(f"  Predicted trajectory: {stab_result['prediction']}")
print()
print("  Class probabilities :")
for cls, prob in sorted(stab_result["probabilities"].items(), key=lambda x: -x[1]):
    bar = "█" * int(prob * 36)
    print(f"    {cls:18s}  {bar:<36s}  {prob:.3f}")


---
## Final Summary — All Predictions


In [ ]:
# ── Visual summary of all pipeline outputs ───────────────────────────
fig = plt.figure(figsize=(16, 7), facecolor='#111111')
gs  = gridspec.GridSpec(2, 4, figure=fig, hspace=0.55, wspace=0.4)

COLOR_OK  = "#2ecc71"
COLOR_BAD = "#e74c3c"
COLOR_WARN= "#f39c12"
COLOR_NEU = "#3498db"

def add_result_box(ax, title, value, subtitle="", color=COLOR_NEU):
    ax.set_facecolor('#1a1a2e')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')
    ax.add_patch(plt.Rectangle((0.03, 0.03), 0.94, 0.94, fill=False,
                                edgecolor=color, linewidth=2.5, transform=ax.transAxes))
    ax.text(0.5, 0.82, title, ha='center', va='center', fontsize=10,
            color='#aaaaaa', transform=ax.transAxes)
    ax.text(0.5, 0.50, value, ha='center', va='center', fontsize=15,
            color=color, fontweight='bold', transform=ax.transAxes)
    ax.text(0.5, 0.20, subtitle, ha='center', va='center', fontsize=8,
            color='#888888', transform=ax.transAxes)

# Row 0 -- imaging results
ax0 = fig.add_subplot(gs[0, 0])
mri_color = COLOR_BAD if MRI_ATROPHY else COLOR_OK
add_result_box(ax0, "MRI Atrophy",
               "ATROPHIC" if MRI_ATROPHY else "HEALTHY",
               f"prob = {mri_avg:.3f}  |  thr = {mri_thr:.3f}", mri_color)

ax1 = fig.add_subplot(gs[0, 1])
pet_color = COLOR_BAD if AMYLOID_STATUS else COLOR_OK
add_result_box(ax1, "PET Amyloid",
               "POSITIVE" if AMYLOID_STATUS else "NEGATIVE",
               f"prob = {pet_avg:.3f}  |  thr = {pet_thr:.3f}", pet_color)

# Clinical summary panel
ax2 = fig.add_subplot(gs[0, 2:])
ax2.set_facecolor('#1a1a2e'); ax2.axis('off')

_mh4  = float(patient_record.get("MH4CARD")  or 0)
_mh9  = float(patient_record.get("MH9ENDO")  or 0)
_mhps = float(patient_record.get("MHPSYCH")  or 0)

clinical_items = [
    ("MMSE",    cur_mmse, 30, 24,  "/ 30",  True),
    ("MoCA",    cur_moca, 30, 26,  "/ 30",  True),
    ("FAQ",     cur_faq,  30,  5,  "/ 30",  False),
    ("HTA",     _mh4,      1,  0.5, "flag", False),
    ("Diabete", _mh9,      1,  0.5, "flag", False),
    ("Psych.",  _mhps,     1,  0.5, "flag", False),
]
ax2.text(0.5, 0.95, "Key Clinical Summary", ha='center', va='top',
         color='white', fontsize=11, fontweight='bold', transform=ax2.transAxes)
for i, (name, val, max_v, thr, unit, low_bad) in enumerate(clinical_items):
    y = 0.78 - i * 0.125
    abnormal = (val < thr) if low_bad else (val > thr)
    c = COLOR_BAD if abnormal else COLOR_OK
    frac = min(val / max_v, 1.0) if max_v else 0
    ax2.barh(y, frac, height=0.07, left=0.22, color=c, alpha=0.7, transform=ax2.transAxes)
    ax2.text(0.21, y + 0.035, f"{name:8s}", ha='right', va='center',
             color='#aaaaaa', fontsize=8.5, transform=ax2.transAxes)
    lbl = ("present" if val >= 1 else "absent") if unit == "flag" else f"{val:.1f}  {unit}"
    ax2.text(0.22 + frac + 0.01, y + 0.035, lbl, ha='left', va='center',
             color=c, fontsize=8.5, transform=ax2.transAxes)

# Row 1 -- prediction results
diag_pred = diag_result["prediction"]
diag_col  = {"MCI": COLOR_WARN, "CN": COLOR_OK, "Dementia": COLOR_BAD}.get(diag_pred, COLOR_NEU)
ax3 = fig.add_subplot(gs[1, 0:2])
diag_probs_sorted = sorted(diag_result["probabilities"].items(), key=lambda x: -x[1])
add_result_box(ax3, "DIAGNOSIS (LightGBM -- 93% F1)",
               diag_pred,
               "  |  ".join(f"{c}: {p:.2f}" for c, p in diag_probs_sorted),
               diag_col)

stab_pred = stab_result["prediction"]
stab_col  = {"MCI_converting": COLOR_BAD, "CN": COLOR_OK,
             "MCI_stable": COLOR_WARN}.get(stab_pred, COLOR_NEU)
ax4 = fig.add_subplot(gs[1, 2:])
stab_probs_sorted = sorted(stab_result["probabilities"].items(), key=lambda x: -x[1])
add_result_box(ax4, "MCI STABILITY (RNN -- 83% acc)",
               stab_pred.replace("_", " "),
               "  |  ".join(f"{c.replace('_',' ')}: {p:.2f}" for c, p in stab_probs_sorted),
               stab_col)

fig.suptitle("MULTIMODAL MCI PREDICTION SYSTEM -- PATIENT REPORT",
             color='white', fontsize=14, fontweight='bold', y=1.01)
plt.savefig("demo_summary.png", dpi=120, bbox_inches='tight', facecolor='#111111')
plt.show()

age_disp = patient_record.get("age")
sex_disp = patient_record.get("PTGENDER")
edu_disp = patient_record.get("PTEDUCAT")
age_disp = age_disp if age_disp is not None else "N/A"
sex_disp = sex_disp if sex_disp is not None else "N/A"
edu_disp = edu_disp if edu_disp is not None else "N/A"

print()
print("=" * 62)
print(f"  FINAL REPORT -- Patient  (Age: {age_disp}, Sex: {sex_disp}, Education: {edu_disp} yrs)")
print("=" * 62)
print(f"  MRI   -> {'Atrophic (MTA=1)' if MRI_ATROPHY else 'Healthy (MTA=0)'}")
print(f"  PET   -> {'Amyloid POSITIVE (A+)' if AMYLOID_STATUS else 'Amyloid NEGATIVE (A-)'}")
print(f"  DIAG  -> {diag_pred}   (p={max(p for _,p in diag_probs_sorted):.3f})")
print(f"  STAB  -> {stab_pred}   (p={max(p for _,p in stab_probs_sorted):.3f})")
print("=" * 62)